In [ ]:
%pip install -q --no-cache-dir --force-reinstall --no-deps https://github.com/santoshcheethiralame-dot/MIRROR/archive/refs/heads/main.zip
%pip install -q bitsandbytes accelerate hf_transfer wordfreq

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)

In [ ]:
import os, urllib.request
os.makedirs("data/concepts", exist_ok=True)
base = "https://raw.githubusercontent.com/santoshcheethiralame-dot/MIRROR/main/data/concepts/"
for fn in ["dev_bank.yaml", "synonyms.yaml", "contexts.yaml"]:
    urllib.request.urlretrieve(base + fn, f"data/concepts/{fn}")
    urllib.request.urlretrieve(base + fn, fn)

In [ ]:
import glob, os
from mirror.hf_model import load_hf

configs = glob.glob("/kaggle/input/**/config.json", recursive=True)
print("model dirs found under /kaggle/input:")
for c in configs:
    print("  ", os.path.dirname(c))
gemma = [os.path.dirname(c) for c in configs if "gemma" in c.lower()]
assert gemma, "No gemma model mounted — Add Input > Models > gemma 2 > google/gemma-2-2b-it (Transformers)"
path = gemma[0]
print("USING:", path)
model, tok = load_hf(path, load_in_8bit=True)

In [ ]:
from mirror.concepts import load_bank
from mirror.forced_choice import collect_forced_choice_hf

INTROSPECTIVE = (
    "<start_of_turn>user\n"
    "I have injected a thought about one concept into your mind. "
    "Which one was it? Choose exactly one word from this list: {options}. "
    "Answer with only that one word."
    "<end_of_turn>\n<start_of_turn>model\n"
)
NEUTRAL = (
    "<start_of_turn>user\n"
    "Choose exactly one word from this list: {options}. "
    "Answer with only that one word."
    "<end_of_turn>\n<start_of_turn>model\n"
)
NAMES = ["elephant", "spider", "eagle", "dolphin", "volcano", "desert", "library", "harbor",
         "joy", "fear", "jealousy", "serenity", "violin", "umbrella", "telescope", "candle"]
bank = load_bank("dev_bank.yaml")

print("=== CONDITION A: introspective framing ===")
result_intro = collect_forced_choice_hf(model, tok, bank, NAMES, layer=13, alpha=1.0,
                                        template=INTROSPECTIVE, answer_marker="model\n",
                                        n_orders=6, n_pairs=12, max_new_tokens=8,
                                        out="forced_intro.jsonl")
print("=== CONDITION B: neutral framing (control) ===")
result_neutral = collect_forced_choice_hf(model, tok, bank, NAMES, layer=13, alpha=1.0,
                                          template=NEUTRAL, answer_marker="model\n",
                                          n_orders=6, n_pairs=12, max_new_tokens=8,
                                          out="forced_neutral.jsonl")

In [ ]:
import numpy as np

from mirror.forced_choice import build_features, concept_abstractness, concept_frequencies
from mirror.prior_null import fit, gamma_ci, gamma_difference_ci

freqs = concept_frequencies(NAMES)
abstract = concept_abstractness(bank, NAMES)


def summarise(name, result):
    records = result["records"]
    unparsed = sum(r["chosen"] is None for r in records)
    hit = np.mean([r["chosen"] == r["concept"] for r in records if r["chosen"]])
    X, y = build_features(NAMES, records, freqs, abstract)
    fitted = fit(X, y)
    lo, hi = gamma_ci(X, y, n_boot=200, rng=np.random.default_rng(0))
    print(f"{name}")
    print(f"  unparseable {unparsed}/{len(records)}   hit rate {hit:.3f} (chance {1/len(NAMES):.3f})")
    print(f"  gamma {fitted.gamma:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")
    return X, y, fitted.gamma


Xa, ya, ga = summarise("A: introspective framing ('which was injected?')", result_intro)
print()
Xb, yb, gb = summarise("B: neutral framing ('pick any word') [CONTROL]", result_neutral)

lo, hi = gamma_difference_ci(Xa, ya, Xb, yb, n_boot=200, rng=np.random.default_rng(1))
print()
print(f"GAMMA DIFFERENCE (A - B): {ga - gb:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")
if lo > 0:
    print("-> EXCLUDES 0: the introspective framing adds signal beyond steering.")
    print("   That difference is the introspective component.")
else:
    print("-> INCLUDES 0: the introspective framing adds nothing.")
    print("   Gamma in condition A is explained by output steering, not access.")